# ESSE 4360 — AI for Payloads · Notebook 2
## Beyond RGB: what the 13 Sentinel-2 bands add (EuroSAT multispectral)

**What you will do**
1. Download the **multispectral EuroSAT** archive: the same 27 000 tiles with all **13 Sentinel-2 bands** (about 2 GB; Colab downloads it in a minute or two).
2. Plot real **spectral signatures** for each land-cover class.
3. Compare a **random forest** that sees only RGB with one that sees all 13 bands, and ask *which bands matter*.
4. Train the same **small CNN** on 3 bands and on 13 bands.

**Runs on:** Google Colab (recommended because of the 2 GB download) or a laptop with enough disk space.

In [ ]:
!pip -q install tifffile
import os, io, time, random, zipfile, urllib.request
import numpy as np, matplotlib.pyplot as plt, tifffile
import torch, torch.nn as nn

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('Running on:', device)
DATA, FIG = 'data', 'figures'; os.makedirs(DATA, exist_ok=True); os.makedirs(FIG, exist_ok=True)

N_PER_CLASS = 300                      # tiles per class to read from the archive
CNN_EPOCHS = 10 if device.type == 'cpu' else 20

# Band order inside the EuroSAT .tif files (as used by TorchGeo) and centre wavelengths (micrometres)
BANDS = ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09', 'B10', 'B11', 'B12', 'B8A']
WL = {'B01': .443, 'B02': .490, 'B03': .560, 'B04': .665, 'B05': .705, 'B06': .740, 'B07': .783, 'B08': .842,
      'B8A': .865, 'B09': .945, 'B10': 1.375, 'B11': 1.610, 'B12': 2.190}

## 1. Download and read a subset
The archive is read directly from the zip file, so nothing large is unpacked on disk.

In [ ]:
URL = 'https://hf.co/datasets/torchgeo/eurosat/resolve/1ce6f1bfb56db63fd91b6ecc466ea67f2509774c/EuroSATallBands.zip'
ZIP = os.path.join(DATA, 'EuroSATallBands.zip')
if not os.path.exists(ZIP):
    print('Downloading ~2 GB ...'); t0 = time.time()
    urllib.request.urlretrieve(URL, ZIP); print(f'done in {time.time() - t0:.0f} s')

z = zipfile.ZipFile(ZIP)
tifs = [n for n in z.namelist() if n.lower().endswith('.tif')]
CLASSES = sorted({n.split('/')[-2] for n in tifs})
print(len(tifs), 'tiles;', CLASSES)

X, y = [], []
for c, name in enumerate(CLASSES):
    members = [n for n in tifs if n.split('/')[-2] == name]
    for n in random.sample(members, min(N_PER_CLASS, len(members))):
        a = tifffile.imread(io.BytesIO(z.read(n)))
        if a.shape[0] == 13: a = a.transpose(1, 2, 0)          # -> (64, 64, 13)
        X.append(a.astype(np.float32) / 10000.0); y.append(c)  # digital numbers -> reflectance
X, y = np.stack(X), np.array(y)
print('X', X.shape, ' reflectance range', X.min().round(3), X.max().round(3))

## 2. Real spectral signatures
Median reflectance of every class in every band. Compare with the typical curves from the lecture: the red edge of vegetation, dark water in the near-infrared, bright bare ground in the short-wave infrared.

In [ ]:
order = np.argsort([WL[b] for b in BANDS])
wl = np.array([WL[b] for b in BANDS])[order]
spec = np.stack([np.median(X[y == c].mean((1, 2)), 0)[order] for c in range(len(CLASSES))])
fig, ax = plt.subplots(figsize=(10, 5))
for c in range(len(CLASSES)):
    ax.plot(wl, spec[c], 'o-', label=CLASSES[c], lw=1.8)
for b0, b1, t in [(0.65, 0.68, 'red'), (0.83, 0.88, 'NIR'), (1.36, 1.39, 'cirrus'), (1.57, 1.65, 'SWIR')]:
    ax.axvspan(b0, b1, color='0.9', zorder=0); ax.text((b0 + b1) / 2, ax.get_ylim()[1] * 0.97, t, ha='center', fontsize=9)
ax.set_xlabel('wavelength (µm)'); ax.set_ylabel('TOA reflectance (median of tile means)')
ax.legend(fontsize=8, ncol=2); ax.set_title('EuroSAT spectral signatures (Sentinel-2 L1C)')
plt.tight_layout(); plt.savefig(f'{FIG}/01_spectra.png', dpi=150); plt.show()

In [ ]:
# Two hand-made indices, computed per tile
def band(name): return X[..., BANDS.index(name)].mean((1, 2))
ndvi = (band('B08') - band('B04')) / (band('B08') + band('B04') + 1e-6)
ndwi = (band('B03') - band('B08')) / (band('B03') + band('B08') + 1e-6)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].boxplot([ndvi[y == c] for c in range(len(CLASSES))]); ax[0].set_title('NDVI per class')
ax[1].boxplot([ndwi[y == c] for c in range(len(CLASSES))]); ax[1].set_title('NDWI (green vs NIR) per class')
for a in ax: a.set_xticks(range(1, len(CLASSES) + 1)); a.set_xticklabels(CLASSES, rotation=60)
plt.tight_layout(); plt.savefig(f'{FIG}/02_indices.png', dpi=150); plt.show()

## 3. Random forest: RGB only versus all 13 bands
Features: the mean and standard deviation of each band in a tile. Same classifier, same split — only the bands change.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
i_tr, i_tmp = train_test_split(np.arange(len(y)), test_size=0.30, stratify=y, random_state=SEED)
i_va, i_te = train_test_split(i_tmp, test_size=0.50, stratify=y[i_tmp], random_state=SEED)

def feats(Xs, bands):
    k = [BANDS.index(b) for b in bands]
    return np.concatenate([Xs[..., k].mean((1, 2)), Xs[..., k].std((1, 2))], 1)

RGB = ['B04', 'B03', 'B02']
res = {}
for name, bands in [('RGB (3 bands)', RGB), ('All 13 bands', BANDS)]:
    Fm = feats(X, bands)
    rf = RandomForestClassifier(400, n_jobs=-1, random_state=SEED).fit(Fm[i_tr], y[i_tr])
    res[name] = accuracy_score(y[i_te], rf.predict(Fm[i_te])); print(f'Random forest, {name:14s}: {res[name] * 100:.1f} %')

# Which bands mattered? (importance of the band-mean features, all-band model)
imp = rf.feature_importances_[:13]
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar([BANDS[i] for i in order], imp[order], color='C0'); ax.set_ylabel('importance'); ax.set_title('Random forest: importance of each band mean')
plt.tight_layout(); plt.savefig(f'{FIG}/03_band_importance.png', dpi=150); plt.show()

## 4. The same small CNN on 3 bands and on 13 bands
Each band is standardised with the training-set mean and standard deviation.

In [ ]:
# ---------- a small, reusable training loop (used by every network below) ----------
def augment(x):
    """Random flips and 90-degree rotations: valid for nadir-looking imagery."""
    if torch.rand(1) < 0.5: x = x.flip(-1)
    if torch.rand(1) < 0.5: x = x.flip(-2)
    return torch.rot90(x, int(torch.randint(0, 4, (1,))), (-2, -1))


def train_model(model, Xtr, ytr, Xva, yva, epochs=8, lr=1e-3, bs=64, aug=True):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    hist = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    Xtr_t, ytr_t = torch.as_tensor(Xtr), torch.as_tensor(ytr)
    for ep in range(epochs):
        model.train(); t0 = time.time(); perm = torch.randperm(len(Xtr_t)); tot = 0
        for i in range(0, len(perm), bs):
            idx = perm[i:i + bs]
            xb = Xtr_t[idx].to(device); yb = ytr_t[idx].to(device)
            if aug: xb = augment(xb)
            opt.zero_grad(); loss = lossf(model(xb), yb); loss.backward(); opt.step()
            tot += loss.item() * len(idx)
        vl, va = evaluate(model, Xva, yva)
        hist['train_loss'].append(tot / len(perm)); hist['val_loss'].append(vl); hist['val_acc'].append(va)
        print(f'epoch {ep + 1:2d}  train loss {tot / len(perm):.3f}  val loss {vl:.3f}  val acc {va * 100:.1f} %  ({time.time() - t0:.0f} s)')
    return model, hist


@torch.no_grad()
def predict(model, X, bs=256):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        out.append(model(torch.as_tensor(X[i:i + bs]).to(device)).softmax(1).cpu())
    return torch.cat(out).numpy()


def evaluate(model, X, y):
    p = predict(model, X)
    loss = nn.functional.nll_loss(torch.log(torch.as_tensor(p) + 1e-9), torch.as_tensor(y)).item()
    return loss, (p.argmax(1) == y).mean()


def plot_history(hist, title, fname):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(hist['train_loss'], 'o-', label='train'); ax[0].plot(hist['val_loss'], 'o-', label='validation')
    ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend(); ax[0].set_title(title)
    ax[1].plot(np.array(hist['val_acc']) * 100, 'o-', color='C2'); ax[1].set_xlabel('epoch'); ax[1].set_ylabel('validation accuracy (%)')
    plt.tight_layout(); plt.savefig(f'{FIG}/{fname}', dpi=150); plt.show()


class SmallCNN(nn.Module):
    """Three conv blocks (conv -> batch norm -> ReLU -> 2x2 max pool), then global average pooling."""
    def __init__(self, n_in=3, n_classes=10, width=16):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(n_in, width), block(width, 2 * width), block(2 * width, 4 * width))  # 64 -> 8 px
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(4 * width, n_classes))

    def forward(self, x):
        return self.head(self.features(x))


def prep(bands):
    k = [BANDS.index(b) for b in bands]
    A = X[..., k].transpose(0, 3, 1, 2)
    mu = A[i_tr].mean((0, 2, 3), keepdims=True); sd = A[i_tr].std((0, 2, 3), keepdims=True) + 1e-6
    return ((A - mu) / sd).astype(np.float32)

for name, bands in [('RGB (3 bands)', RGB), ('All 13 bands', BANDS)]:
    A = prep(bands); torch.manual_seed(SEED)
    print(f'--- SmallCNN, {name} ---')
    m, h = train_model(SmallCNN(len(bands), len(CLASSES)), A[i_tr], y[i_tr], A[i_va], y[i_va], epochs=CNN_EPOCHS)
    res['CNN, ' + name] = accuracy_score(y[i_te], predict(m, A[i_te]).argmax(1))
    print(f"test accuracy {res['CNN, ' + name] * 100:.1f} %")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(list(res), [v * 100 for v in res.values()], color=['0.6', '0.35', 'C0', 'C1']); ax.set_xlim(0, 100); ax.set_xlabel('test accuracy (%)')
for i, v in enumerate(res.values()): ax.text(v * 100 + 1, i, f'{v * 100:.1f}', va='center')
ax.invert_yaxis(); plt.tight_layout(); plt.savefig(f'{FIG}/04_rgb_vs_13.png', dpi=150); plt.show()

## 5. Questions
1. Which bands does the random forest rely on? Do they match the physics (red edge, NIR, SWIR)?
2. B10 (cirrus, 1.375 µm) barely reaches the ground. What is it good for, and why is it unimportant here?
3. How much do the extra bands help the random forest? The CNN? Why might they help one more than the other?
4. **Payload question:** a CubeSat imager can afford only four bands. Using the importance plot, which four would you choose for land-cover mapping, and why?

In [ ]:
with zipfile.ZipFile('figures_notebook2.zip', 'w') as zz:
    for f in sorted(os.listdir(FIG)): zz.write(os.path.join(FIG, f))
try:
    from google.colab import files; files.download('figures_notebook2.zip')
except ImportError:
    print('Saved figures_notebook2.zip')